#### Extraction des caractéristiques avec model_vgg16_fullyConnected_sans_augm_couche_gelee_batch_16.pt

Ce script extrait les caractéristiques (features) des images satellites d'un nouvel ensemble de données en utilisant un modèle VGG16 pré-entraîné avec des couches gelées. Il reproduit exactement la même architecture que celle utilisée pendant l'entraînement, avec les couches convolutives et les premières couches du classifier gelées, pour assurer une cohérence dans l'extraction des caractéristiques.
Le script charge les images, les prétraite avec les mêmes transformations que pendant l'entraînement (redimensionnement à 224x224 pixels et normalisation), et extrait un vecteur de 4096 caractéristiques pour chaque image. Ces caractéristiques sont extraites de l'avant-dernière couche du réseau, juste avant la couche de classification finale.
Une gestion robuste des erreurs est implémentée pour traiter les cas d'images manquantes ou corrompues, en les remplaçant par des vecteurs de zéros. Le script utilise tqdm pour afficher une barre de progression et fournit des statistiques détaillées sur le processus d'extraction.
Finalement, les caractéristiques extraites sont combinées avec les données originales dans un DataFrame et sauvegardées dans un fichier CSV. Le script maintient la traçabilité en utilisant des noms de fichiers cohérents avec le modèle d'origine, incluant les paramètres de configuration comme la taille du batch et l'état des couches (gelées).

In [ ]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models
from tqdm import tqdm

# Chemins des fichiers et répertoires
model_path = r"D:\wealth_predict_sentinel\models\model_vgg16_fullyConnected_sans_augm_couche_gelee_batch_16.pt"
test_image_dir = r"D:\wealth_predict_sentinel\Data\downloaded\Image_satellite_base_EHCVM_2018_Zoom_14_Sentinel_2_pour_an_2023" # menage EHCVM 2018 leurs images en 2023
csv_path = r"D:\wealth_predict_sentinel\Data\processed_csv\Data_EHCVM_2018_with_images_names.csv"
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConnected_sans_augm_couche_gelee_batch_16_base_EHCVM_2018_pour_an_2023.csv" #On met à jour ce nom quand on met de nouvelles données

# Définir l'architecture exactement comme dans l'entraînement
class VGG16Standard(nn.Module):
    def __init__(self, num_classes=4):
        super(VGG16Standard, self).__init__()
        self.vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        # Geler les couches convolutives
        for param in self.vgg16.features.parameters():
            param.requires_grad = False
        
        # Geler les premières couches du classifier
        for param in list(self.vgg16.classifier.children())[:-2]:
            param.requires_grad = False
        
        # Modifier la dernière couche
        num_features = self.vgg16.classifier[-1].in_features
        self.vgg16.classifier[-1] = nn.Linear(num_features, num_classes)

    def get_features(self, x):
        # Passer à travers les couches convolutives
        x = self.vgg16.features(x)
        x = self.vgg16.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Extraire les caractéristiques de l'avant-dernière couche (4096)
        classifier = list(self.vgg16.classifier.children())
        for layer in classifier[:-1]:  # On s'arrête avant la dernière couche
            x = layer(x)
        return x

# Transformation des images (identique à l'entraînement)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Fonction pour charger une image
def load_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        return transform(image)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

print("Configuration du device...")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device: {device}")

print("Chargement du modèle...")
model = VGG16Standard(num_classes=4).to(device)
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Vérification des dimensions
print("Vérification des dimensions des caractéristiques...")
dummy_image = torch.randn(1, 3, 224, 224).to(device)
dummy_features = model.get_features(dummy_image)
print(f"Dimensions des caractéristiques : {dummy_features.shape}")  # Devrait être [1, 4096]

print("Chargement des données...")
df = pd.read_csv(csv_path)
total_images = len(df)
print(f"Nombre total d'images à traiter : {total_images}")

# Extraction des caractéristiques
print("Début de l'extraction des caractéristiques...")
features_list = []

for idx, row in tqdm(df.iterrows(), total=total_images, desc="Extraction des caractéristiques"):
    image_name = row['nom de l\'image']
    image_path = os.path.join(test_image_dir, image_name)
    
    if os.path.exists(image_path):
        image_tensor = load_image(image_path)
        if image_tensor is not None:
            # Extraction des caractéristiques
            with torch.no_grad():
                image_tensor = image_tensor.unsqueeze(0).to(device)
                features = model.get_features(image_tensor)
                features = features.cpu().numpy().flatten()
        else:
            features = np.zeros(4096)
            print(f"Image invalide : {image_name}")
    else:
        features = np.zeros(4096)
        print(f"Image introuvable : {image_name}")
    
    features_list.append(features)

# Création du DataFrame avec les caractéristiques
print("Création du DataFrame avec les caractéristiques...")
features_df = pd.DataFrame(features_list, columns=[f"feature_{i}" for i in range(4096)])

# Combinaison avec le DataFrame original
print("Combinaison avec les données originales...")
df_with_features = pd.concat([df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)

# Sauvegarde des résultats
print(f"Sauvegarde du DataFrame sous : {output_path}")
df_with_features.to_csv(output_path, index=False)

print("\nStatistiques finales:")
print(f"Nombre total d'images traitées : {total_images}")
print(f"Dimensions du DataFrame final : {df_with_features.shape}")
print("Terminé!")

In [ ]:
pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConnected_sans_augm_couche_gelee_batch_16_base_EHCVM_2018_pour_an_2023.csv")